## Versioning , Monitoring and Automated Test Suits

Problem Statement
Need to have analysis on the guardRail performance and input data ,
Need to to deploy new version of GuardRail ,
Need to compare performance of the GuardRail with other time frame.


In [ ]:
#Basic Setup

import os
import boto3
import botocore
import json

aws_region = "us-east-1"
guardrail_id = os.environ.get("BEDROCK_GUARDRAIL_ID")
logging_role_arn = os.environ.get("BEDROCK_LOGGING_ROLE_ARN")
guardrail_version = os.environ.get("BEDROCK_GUARDRAIL_VERSION", "DRAFT")

# Created the Bedrock client
bedrock_client = boto3.client(
    service_name="bedrock",
    region_name=aws_region,
)

# Created the Bedrock runtime client
bedrock_runtime_client = boto3.client(
    service_name="bedrock-runtime",
    region_name=aws_region
    )

# Print the URL for the Bedrock client
print(bedrock_client.meta.endpoint_url)
print(bedrock_runtime_client.meta.endpoint_url)


## Creating Version of the GuardRail

In [ ]:
def create_guardrail_version(guardrail_id , description="This is a new version of the guardrail"):
    version_response = bedrock_client.create_guardrail_version(
    guardrailIdentifier=guardrail_id,   # your guardrail ID
    description=description,  # optional
    )

    print(version_response["guardrailId"])
    print(version_response["version"])  # e.g. "1", "2", "3"

In [ ]:
create_guardrail_version(guardrail_id,"Initial Version of the GuardRail")


## Editing the Already Existing GuardRail


In [ ]:
def edit_guardrail(guardrail_id):
    response = bedrock_client.update_guardrail(
        guardrailIdentifier=guardrail_id,

        name="Grounding_GuardRail_Protection",
        description="This is a new version of the guardrail",
        blockedInputMessaging=(
            "The input was blocked because it contains "
            "content that violates the safety policy."
        ),

        blockedOutputsMessaging=(
            "The generated response was blocked because it "
            "contains content that violates the safety policy."
        ),

        contentPolicyConfig={
            "filtersConfig": [
                {
                    "type": "HATE",

                    # Check user input
                    "inputStrength": "HIGH",

                    # Check LLM output
                    "outputStrength": "HIGH",

                    "inputAction": "BLOCK",
                    "outputAction": "BLOCK",

                    "inputEnabled": True,
                    "outputEnabled": True
                }
            ]
        }
    )

    print(response)

In [ ]:
edit_guardrail(guardrail_id)


**Printing the Policy of DRAFT and V1 GuardRails version**

In [ ]:
draft = bedrock_client.get_guardrail(
    guardrailIdentifier=guardrail_id,
    guardrailVersion="DRAFT",
)

v1 = bedrock_client.get_guardrail(
    guardrailIdentifier=guardrail_id,
    guardrailVersion="1",
)

print("=" * 60)
print("DRAFT GUARDRAIL")
print("=" * 60)
print("Name       :", draft.get("name"))
print("Version    :", draft.get("version"))
print("Description:", draft.get("description"))
print("-" * 60)
print("Grounding Policy:")
print(json.dumps(draft.get("contextualGroundingPolicy"), indent=2))
print("Content Policy:")
print(json.dumps(draft.get("contentPolicy"), indent=2))
print("=" * 60)
print("================================================================")
print("VERSION 1 GUARDRAIL")
print("=" * 60)
print("Name       :", v1.get("name"))
print("Version    :", v1.get("version"))
print("Description:", v1.get("description"))
print("-" * 60)
print("Grounding Policy:")
print(json.dumps(v1.get("contextualGroundingPolicy"), indent=2))
print("Content Policy:")
print(json.dumps(v1.get("contentPolicy"), indent=2))
print("=" * 60)


## Logging the Traffic to Cloudwatch

In [ ]:
from botocore.exceptions import ClientError

def enable_guardrail_cloudwatch_logging(
    guardrail_id,
    region="us-east-1",
    logging_role_arn=None,  # required IAM role Bedrock can assume
):
    """
    Enable Bedrock model invocation logging to CloudWatch.
    Guardrail-specific filtering is done in CloudWatch Logs Insights.
    """

    bedrock = bedrock_client
    logs = boto3.client("logs", region_name=region)

    # 1) Verify guardrail exists
    bedrock.get_guardrail(
        guardrailIdentifier=guardrail_id,
        guardrailVersion="DRAFT",
    )

    # 2) Create dedicated log group
    log_group_name = f"/aws/bedrock/guardrails/{guardrail_id}"

    try:
        logs.create_log_group(logGroupName=log_group_name)
        print(f"Created log group: {log_group_name}")
    except ClientError as e:
        if e.response["Error"]["Code"] != "ResourceAlreadyExistsException":
            raise
        print(f"Log group already exists: {log_group_name}")

    if not logging_role_arn:
        raise ValueError(
            "logging_role_arn is required. "
            "Create an IAM role Bedrock can assume with logs:CreateLogStream and logs:PutLogEvents."
        )

    # 3) Enable Bedrock model invocation logging
    response = bedrock.put_model_invocation_logging_configuration(
        loggingConfig={
            "cloudWatchConfig": {
                "logGroupName": log_group_name,
                "roleArn": logging_role_arn,
            },
            "textDataDeliveryEnabled": True,
            "imageDataDeliveryEnabled": False,
            "embeddingDataDeliveryEnabled": False,
        }
    )

    print("CloudWatch logging enabled for Bedrock invocations")
    print("Log group:", log_group_name)

    return {
        "guardrail_id": guardrail_id,
        "log_group_name": log_group_name,
        "logging_role_arn": logging_role_arn,
        "response": response,
    }

In [ ]:
# Need to look and solve this issue
result = enable_guardrail_cloudwatch_logging(
    guardrail_id=guardrail_id,
    region=aws_region,
    logging_role_arn=logging_role_arn,
)

## Versioning Checklist

- [x] Create GuardRail
- [x] Make version of GuardRail
- [ ] Edit GuardRail
- [ ] Rollback GuardRail
- [ ] Remove policy from GuardRails
- [ ] How to copy a already exisiting GuardRails and Create a new copy from it
